This notebook converts the final labeled job dataset into binary skill vectors using a global skill vocabulary.
The resulting dataset is used for machine learning–based skill-to-role classification, skill gap analysis, and career path simulation.

In [36]:
import pandas as pd
import numpy as np
from pathlib import Path


In [37]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [38]:
# Paths
DATA_DIR = Path("data/processed")

JOBS_FILE = DATA_DIR / "jobs_labeled.csv"
SKILLS_FILE = DATA_DIR / "skills.csv"

OUTPUT_FILE = DATA_DIR / "job_skill_vectors.csv"


In [39]:
jobs_df = pd.read_csv(JOBS_FILE)
skills_df = pd.read_csv(SKILLS_FILE)


In [40]:
print("Jobs dataset shape:", jobs_df.shape)
print("Skills dataset shape:", skills_df.shape)

jobs_df.head()


Jobs dataset shape: (5470, 17)
Skills dataset shape: (300, 5)


,job_uid,source,job_title,description,requirements_text,job_text,matched_skill_ids,matched_skills,matched_skill_count,experience_raw,salary_raw,job_type,industry,title_clean_for_role,role_id,role_title,job_title_clean
0,JOBS_000001,jobs.csv,"(Jr/Sr) QA Engineer, KMS Labs - BONUS",KMS Labs is the startup incubation arm of KMS ...,QA QC English Tester,KMS Labs is the startup incubation arm of KMS ...,SK161;SK239;SK007;SK028;SK206;SK041;SK068;SK11...,agile;computer science;mobile;problem solving;...,10,NaN,NaN,NaN,NaN,qa engineer,JR_QA_ENG,QA Engineer (Entry Level),QA Engineer (Entry Level)
1,JOBS_000006,jobs.csv,Sr IT Business Analyst (English) - BONUS,Full 13th Month Salary. ***Apply and Join in S...,Business Analyst Agile English,Full 13th Month Salary. ***Apply and Join in S...,SK161;SK067;SK148;SK225;SK102;SK041;SK122;SK050,agile;business;business requirements;it;projec...,8,NaN,NaN,NaN,NaN,sr it business analyst,JR_BUSINESS_ANALYST,Business Analyst (Entry Level),Business Analyst (Entry Level)
2,JOBS_000009,jobs.csv,Senior Data Engineer - BONUS,Full 13th Month Salary. ***Apply and Join in S...,Database SQL English,Full 13th Month Salary. ***Apply and Join in S...,SK161;SK067;SK145;SK124;SK253;SK129;SK012;SK19...,agile;business;business analysis;data;data ana...,16,NaN,NaN,NaN,NaN,senior data engineer,DATA_ENGINEER_INT,Data Engineer (Entry Level),Data Engineer (Entry Level)
3,JOBS_000011,jobs.csv,(Junior/Senior) Test Engineer - BONUS,Full 13th Month Salary ***Apply and Join in Se...,Tester English QA QC,Full 13th Month Salary ***Apply and Join in Se...,SK021;SK225;SK072;SK007;SK028;SK041;SK122;SK28...,api;it;junit;mobile;problem solving;software;s...,10,NaN,NaN,NaN,NaN,test engineer,JR_QA_ENG,QA Engineer (Entry Level),QA Engineer (Entry Level)
4,JOBS_000016,jobs.csv,Principle Data Engineer (Medior/Senior),"Short Description: In this role, you will prod...",Python SQL AWS,"Short Description: In this role, you will prod...",SK161;SK181;SK043;SK124;SK115;SK012;SK024;SK00...,agile;aws;big data;data;data mining;database;d...,19,NaN,NaN,NaN,NaN,principle data engineer,DATA_ENGINEER_INT,Data Engineer (Entry Level),Data Engineer (Entry Level)


In [41]:
skills_df = pd.read_csv(SKILLS_FILE)

global_skill_ids = (
    skills_df["skill_id"]
    .astype(str)
    .str.strip()
    .str.upper()
    .tolist()
)

print("Total skill IDs:", len(global_skill_ids))

Total skill IDs: 300


In [42]:
SKILL_ID_COLUMN = "matched_skill_ids"
LABEL_COLUMN = "role_id"


In [43]:
def normalize_skill_ids(skill_id_cell):
    if pd.isna(skill_id_cell):
        return set()

    return set(
        s.strip().upper()   # ✅ FORCE UPPERCASE
        for s in str(skill_id_cell).split(";")
        if s.strip()
    )


In [44]:
jobs_df["normalized_skill_ids"] = jobs_df["matched_skill_ids"].apply(normalize_skill_ids)

jobs_df[["matched_skill_ids", "normalized_skill_ids"]].head()


,matched_skill_ids,normalized_skill_ids
0,SK161;SK239;SK007;SK028;SK206;SK041;SK068;SK11...,"{SK135, SK239, SK041, SK161, SK068, SK026, SK1..."
1,SK161;SK067;SK148;SK225;SK102;SK041;SK122;SK050,"{SK102, SK041, SK067, SK161, SK050, SK148, SK2..."
2,SK161;SK067;SK145;SK124;SK253;SK129;SK012;SK19...,"{SK124, SK253, SK192, SK102, SK041, SK067, SK0..."
3,SK021;SK225;SK072;SK007;SK028;SK041;SK122;SK28...,"{SK135, SK041, SK286, SK026, SK225, SK122, SK0..."
4,SK161;SK181;SK043;SK124;SK115;SK012;SK024;SK00...,"{SK003, SK225, SK124, SK016, SK043, SK024, SK0..."


In [45]:
def build_skill_id_vector(job_skill_ids, global_skill_ids):
    return [1 if sid in job_skill_ids else 0 for sid in global_skill_ids]

skill_vectors = jobs_df["normalized_skill_ids"].apply(
    lambda s: build_skill_id_vector(s, global_skill_ids)
)


In [46]:
X = pd.DataFrame(
    skill_vectors.tolist(),
    columns=[f"skill_{sid.lower()}" for sid in global_skill_ids]
)



y = jobs_df[LABEL_COLUMN]


In [47]:
final_df = pd.concat(
    [
        jobs_df[["job_title", "job_title_clean", "role_id", "role_title"]],
        X
    ],
    axis=1
)


print(final_df.columns[:10])
print("Label column present:", "role_id" in final_df.columns)

Index(['job_title', 'job_title_clean', 'role_id', 'role_title', 'skill_sk001',
       'skill_sk002', 'skill_sk003', 'skill_sk004', 'skill_sk005',
       'skill_sk006'],
      dtype='object')
Label column present: True


In [48]:
final_df.filter(like="skill_").sum().sort_values(ascending=False).head(20)


final_df.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)



Saved: data/processed/job_skill_vectors.csv


In [49]:
# Check vector correctness
final_df.iloc[0]


,0
job_title,"(Jr/Sr) QA Engineer, KMS Labs - BONUS"
job_title_clean,QA Engineer (Entry Level)
role_id,JR_QA_ENG
role_title,QA Engineer (Entry Level)
skill_sk001,0
...,...
skill_sk296,0
skill_sk297,0
skill_sk298,0
skill_sk299,0


In [50]:
# Check class distribution
final_df["role_id"].value_counts()


,count
role_id,
JR_MOBILE_DEV,1325
JR_SE,742
DEVOPS_TRAINEE,620
JR_BUSINESS_ANALYST,613
DATA_ENGINEER_INT,581
DATA_ANALYST_INT,404
JR_FS_DEV,394
JR_BE_DEV,267
JR_FE_DEV,164
